In [1]:
mutable struct ScopedAssignment{T}
    target::Union{Ref{T}, Nothing}
    backup::T
    
    # Constructor
    function ScopedAssignment(target::Ref{T}, value::T) where T
        backup = target[]  # Get the current value
        target[] = value   # Set the new value
        new{T}(target, backup)
    end
    
    # Constructor for the null case
    ScopedAssignment{T}() where T = new{T}(nothing, zero(T))
end

# Destructor-like functionality using finalizers
function Base.close(sa::ScopedAssignment{T}) where T
    if sa.target !== nothing
        sa.target[] = sa.backup
        sa.target = nothing  # Mark as cleaned up
    end
end

# Context manager interface
function Base.show(io::IO, sa::ScopedAssignment{T}) where T
    print(io, "ScopedAssignment{$T}(target=$(sa.target !== nothing ? sa.target[] : "nothing"), backup=$(sa.backup))")
end

# Helper function to use with 'do' syntax
function with_scoped_assignment(f::Function, target::Ref{T}, value::T) where T
    sa = ScopedAssignment(target, value)
    try
        f()
    finally
        close(sa)
    end
end

with_scoped_assignment (generic function with 1 method)

In [8]:
function example()
    x = Ref(10)  # Create a reference to hold our value
    
    println("Initial value: ", x[])
    
    # Using do-block syntax (recommended way)
    new_thing = 0.0
    with_scoped_assignment(x, 20) do
        println("Inside scope: ", x[])
        # Do some work with the temporarily assigned value
        new_thing = x[] * 25.0
    end
    
    println("After scope: ", x[])
    println("new_thing: ", new_thing)
end

example (generic function with 1 method)

In [9]:
# Running the example:
example()

Initial value: 10
Inside scope: 20
After scope: 10
new_thing: 500.0
